# 07. Lecke - Tervezési Tervezési Minta

Ez a jegyzetfüzet a Microsoft Agent Framework használatával bemutatja az **Tervezési Tervezési Mintát** AI ügynökök számára.
Megtanulod, hogyan bonts komplex utazási kéréseket strukturált alfeladatokra, hogyan rendeld őket szakértő ügynökökhöz,
és hogyan hajtsd végre a kialakult tervet — mindezt Pydantic modellek által vezérelt strukturált kimenettel.


## Beállítás


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Feladat lebontása

A feladat lebontása a tervezési minta magja. Ahelyett, hogy egyetlen ügynök kezeli egy összetett kérést az elejétől a végéig, a problémát kisebb, jól definiált **al-feladatokra** bontjuk. Minden al-feladatot egy specialistának kijelölt ügynökhöz (pl. járatok, szállodák, programok) rendelünk, egyértelmű prioritásokkal és függőségi sorrenddel.

Ez a megközelítés több előnnyel jár:
- **Átláthatóság**: minden al-feladatnak egyetlen felelőssége van.
- **Párhuzamosság**: a független al-feladatok egyidejűleg futhatnak.
- **Megbízhatóság**: a hibák az egyes al-feladatokra korlátozódnak.
- **Költségkövetés**: a költségeket al-feladatonként becsüljük és összegezzük.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Strukturált kimenettel rendelkező tervezőügynök létrehozása

A tervezőügynök **portaszolgálati koordinátorként** működik. Egy magas szintű utazási kérés alapján egy strukturált `TravelPlan`-t hoz létre — lebontva a kérést alsófeladatokra, priorításokat állít be, és azonosítja a függőségeket, hogy egy portás vagy végrehajtó réteg el tudja végezni a munkát.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Egy terv végrehajtása szakértői eszközökkel

Miután az elsődleges ügyintéző elkészítette a strukturált tervet, a **portásügynök** hajtja végre azt.
Minden szakértői eszköz egy alfeladat kategóriáját kezeli (járatok, szállodák, tevékenységek). A portás
a terv alfeladatait függőségi sorrendben iterálja végig, és mindegyiket a megfelelő eszköznek továbbítja.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Összefoglaló

Ebben a leckében megismerted az AI ügynökök **Tervezési mintáját**:

1. **Feladatlebontás** — Egy recepciós tervező ügynök egy összetett utazási kérést strukturált alfeladatokra bont Pydantic modellek segítségével, mindegyiket szakértő ügynökhöz rendelve prioritásokkal és függőségekkel.
2. **Strukturált kimenet** — A `response_format` átadásával az ügynök egy validált `TravelPlan` objektumot ad vissza szabad szöveg helyett, ami megbízhatóvá teszi a további feldolgozást.
3. **Terv végrehajtás** — Egy concierge ügynök végigmegy az alfeladatokon szakértő eszközökkel (`book_flight`, `reserve_hotel`, `book_activity`), hogy végrehajtsa a tervet és jelentse az eredményeket.

Ez a minta szétválasztja *mit kell tenni* (tervezés) és *hogyan kell tenni* (végrehajtás), modulárisabbá, tesztelhetőbbé és könnyebben bővíthetővé téve az ügynököket.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jóváhagyás**:  
Ez a dokumentum az AI fordítószolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti, anyanyelvi dokumentum tekintendő hivatalos forrásnak. Kritikus információk esetén professzionális, emberi fordítást javaslunk. Nem vállalunk felelősséget az ebből eredő félreértésekért vagy félreértelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
